# Py-Code — Aulas e projetos interativos
Professor Thiago Fillipe Soares

Cobrinha, Tetris, RPG e ELIZA: use a instalação na etapa 4 e execute a etapa 5 inteira. Os botões aparecem na saída da célula. Mantenha a sessão conectada.


## O que é Python?
Python é uma linguagem de programação. Um programa é uma sequência de instruções que o computador executa. No Google Colab você escreve Python em células e clica em executar.

### Etapa 1
print() mostra uma mensagem na tela. As aspas marcam um texto.

In [ ]:
print("Olá, mundo!")

### Etapa 2
A variável nome guarda um texto. print(nome) usa o conteúdo da variável.

In [ ]:
nome = "Ana"
print(nome)

### Etapa 3
O mesmo programa pode juntar texto e variável sem complicar o código.

In [ ]:
nome = "Ana"
print("Olá,", nome)

### Desafio
Troque o nome do aluno e faça o programa mostrar uma segunda mensagem.

## Listas Python
Listas guardam vários elementos na mesma variável. Os índices começam em zero. Experimente alterar as frutas.

### Etapa 1
Uma lista aparece entre colchetes. Os três itens são textos.

In [ ]:
frutas = ["maçã", "uva", "banana"]
print(frutas)

### Etapa 2
frutas[0] pega o primeiro elemento. Python começa a contar em zero.

In [ ]:
frutas = ["maçã", "uva", "banana"]
print(frutas[0])

### Etapa 3
append adiciona um elemento. for percorre cada item e print o mostra.

In [ ]:
frutas = ["maçã", "uva"]
frutas.append("banana")
for fruta in frutas:
    print(fruta)

### Desafio
Crie uma lista de três animais, adicione um quarto com append e mostre o segundo item.

## Cálculos e decisões
Programar também é escolher caminhos: if decide, for repete, e os operadores fazem cálculos.

### Etapa 1
+ soma. Guarde os números em variáveis simples para reaproveitar.

In [ ]:
a = 4
b = 3
print(a + b)

### Etapa 2
if verifica uma condição. else é o caminho contrário.

In [ ]:
pontos = 12
if pontos >= 10:
    print("Você passou de fase!")
else:
    print("Continue tentando!")

### Etapa 3
range(1, 6) produz 1, 2, 3, 4, 5. A repetição economiza linhas.

In [ ]:
for numero in range(1, 6):
    print(numero)

### Desafio
Altere a variável pontos para 3 e depois para 15. Explique a diferença de saída.

## Cobrinha
Cobrinha jogável com tabuleiro, alimento, pontuação e colisão. Na etapa 4, instale os widgets; na etapa 5, execute o programa inteiro, clique em Iniciar e use os botões de direção. Os primeiros exemplos continuam ensinando os fundamentos. A interface abre na saída do Colab, com uma sessão conectada; estes códigos não executam dentro do site.

### Etapa 1
Cada tupla (x, y) representa uma parte da cobrinha.

In [ ]:
cobra = [(2, 2), (1, 2), (0, 2)]
print(cobra)

### Etapa 2
A nova cabeça vai para x+1. pop remove a última parte, mantendo o tamanho.

In [ ]:
cobra = [(2, 2), (1, 2), (0, 2)]
x, y = cobra[0]
cobra.insert(0, (x + 1, y))
cobra.pop()
print(cobra)

### Etapa 3
for repete o movimento. O objetivo é compreender a regra antes do desenho.

In [ ]:
cobra = [(2, 2), (1, 2), (0, 2)]
for passo in range(3):
    x, y = cobra[0]
    cobra.insert(0, (x + 1, y))
    cobra.pop()
    print("Passo", passo + 1, cobra)

### Etapa 4
Preparação: execute esta célula no Google Colab. Se houver pedido de reinício, reinicie a sessão e execute a etapa seguinte. Esta instalação é comum aos quatro projetos.

In [ ]:
%pip install -q "ipywidgets>=8.1,<9"

### Etapa 5
Cobrinha jogável com tabuleiro, alimento, pontuação e colisão. Na etapa 4, instale os widgets; na etapa 5, execute o programa inteiro, clique em Iniciar e use os botões de direção. Não use um arquivo .py no terminal para esta interface: ela depende do ambiente de notebook e, nos dois jogos de tabuleiro, do loop assíncrono do Colab.

In [ ]:
import random

class Cobrinha:
    def __init__(self, tamanho=16):
        self.tamanho = tamanho
        self.corpo = [(5, 7), (4, 7), (3, 7)]
        self.direcao = (1, 0)
        self.proxima = self.direcao
        self.pontos = 0
        self.fim = False
        self.comida = self.sortear_comida()

    def sortear_comida(self):
        livres = [(x, y) for x in range(self.tamanho) for y in range(self.tamanho)
                  if (x, y) not in self.corpo]
        return random.choice(livres) if livres else None

    def virar(self, dx, dy):
        if (dx, dy) in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            if (dx, dy) != (-self.direcao[0], -self.direcao[1]):
                self.proxima = (dx, dy)

    def passo(self):
        if self.fim:
            return
        self.direcao = self.proxima
        x, y = self.corpo[0]
        cabeca = (x + self.direcao[0], y + self.direcao[1])
        cresce = cabeca == self.comida
        obstaculos = self.corpo if cresce else self.corpo[:-1]
        if not (0 <= cabeca[0] < self.tamanho and 0 <= cabeca[1] < self.tamanho) or cabeca in obstaculos:
            self.fim = True
            return
        self.corpo.insert(0, cabeca)
        if cresce:
            self.pontos += 1
            self.comida = self.sortear_comida()
            if self.comida is None:
                self.fim = True
        else:
            self.corpo.pop()

if __name__ == "__main__":
    import asyncio
    import ipywidgets as w
    from IPython.display import display
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass

    # Encerra a partida anterior quando a célula é executada novamente.
    if "tarefa_cobrinha" in globals():
        tarefa_cobrinha.cancel()
    jogo = Cobrinha()
    rodando = False
    painel = w.HTML()
    iniciar = w.Button(description="Iniciar / Pausar")
    reiniciar = w.Button(description="Reiniciar")

    def desenhar():
        blocos = ""
        for i, (x, y) in enumerate(jogo.corpo):
            cor = "#a3e635" if i == 0 else "#22c55e"
            blocos += f'<rect x="{x*20+1}" y="{y*20+1}" width="18" height="18" rx="4" fill="{cor}"/>'
        if jogo.comida is not None:
            x, y = jogo.comida
            blocos += f'<circle cx="{x*20+10}" cy="{y*20+10}" r="7" fill="#fb7185"/>'
        estado = "Fim de jogo — Reinicie" if jogo.fim else ("Jogando" if rodando else "Pausado — clique em Iniciar")
        painel.value = f'<p><b>Cobrinha | Pontos: {jogo.pontos}</b> | {estado}</p><svg viewBox="0 0 320 320" width="320" style="max-width:100%;background:#0f172a">{blocos}</svg>'

    def alternar(_):
        global rodando
        if not jogo.fim:
            rodando = not rodando
        desenhar()

    def resetar(_):
        global jogo, rodando
        jogo = Cobrinha()
        rodando = False
        desenhar()

    iniciar.on_click(alternar)
    reiniciar.on_click(resetar)
    botoes = []
    for texto, dx, dy in [("↑", 0, -1), ("←", -1, 0), ("↓", 0, 1), ("→", 1, 0)]:
        b = w.Button(description=texto, layout=w.Layout(width="65px", height="45px"))
        b.on_click(lambda _, dx=dx, dy=dy: jogo.virar(dx, dy))
        botoes.append(b)

    async def atualizar():
        while True:
            await asyncio.sleep(0.28)
            if rodando and not jogo.fim:
                jogo.passo()
                desenhar()

    display(w.VBox([painel, w.HBox(botoes), w.HBox([iniciar, reiniciar])]))
    desenhar()
    tarefa_cobrinha = asyncio.get_running_loop().create_task(atualizar())


### Desafio
Altere o intervalo de asyncio.sleep de 0.28 para 0.4 e compare a velocidade.

## Ping-Pong
O Pong é um ótimo primeiro projeto: posição, velocidade e um teste if para a bola rebater.

### Etapa 1
A posição aumenta quando a velocidade é positiva.

In [ ]:
bola = 0
velocidade = 1
bola = bola + velocidade
print(bola)

### Etapa 2
Primeiro movemos a bola de 4 para 5. Ao alcançar a borda, a velocidade passa a -1. Saída: Posição: 5 e Direção: -1.

In [ ]:
bola = 4
velocidade = 1
limite_direito = 5
bola = bola + velocidade
if bola >= limite_direito:
    bola = limite_direito
    velocidade = -abs(velocidade)
print("Posição:", bola)
print("Direção:", velocidade)

### Etapa 3
A bola rebate nos dois limites. >= e <= também detectam quando um movimento ultrapassa a borda; ajustamos a posição para ficar dentro da quadra. É uma simulação textual, sem raquetes.

In [ ]:
limite_esquerdo = 0
limite_direito = 5
bola = 2
velocidade = 1
for tempo in range(1, 11):
    bola = bola + velocidade
    if bola >= limite_direito:
        bola = limite_direito
        velocidade = -abs(velocidade)
    elif bola <= limite_esquerdo:
        bola = limite_esquerdo
        velocidade = abs(velocidade)
    print("Passo", tempo, "posição", bola, "direção", velocidade)

### Desafio
Na etapa 3, altere limite_direito para 6. Execute novamente e anote em quais passos a velocidade muda de sinal.

## Tetris
Tetris jogável com sete peças, rotação, queda automática, remoção de linhas e pontuação. Execute a instalação da etapa 4 e o programa da etapa 5; clique em Iniciar e use os botões. Os primeiros exemplos continuam ensinando os fundamentos. A interface abre na saída do Colab, com uma sessão conectada; estes códigos não executam dentro do site.

### Etapa 1
Uma peça é representada por duas linhas com dois blocos cada.

In [ ]:
peca = [[1, 1], [1, 1]]
print(peca)

### Etapa 2
Cada sublista é uma linha da peça, percorrida com for.

In [ ]:
peca = [[0, 1, 0], [1, 1, 1]]
for linha in peca:
    print(linha)

### Etapa 3
Transformamos os números em desenho textual. A lógica é curta e funciona no Colab.

In [ ]:
peca = [[0, 1, 0], [1, 1, 1]]
for linha in peca:
    desenho = ""
    for bloco in linha:
        desenho += "[]" if bloco else "  "
    print(desenho)

### Etapa 4
Preparação: execute esta célula no Google Colab. Se houver pedido de reinício, reinicie a sessão e execute a etapa seguinte. Esta instalação é comum aos quatro projetos.

In [ ]:
%pip install -q "ipywidgets>=8.1,<9"

### Etapa 5
Tetris jogável com sete peças, rotação, queda automática, remoção de linhas e pontuação. Execute a instalação da etapa 4 e o programa da etapa 5; clique em Iniciar e use os botões. Não use um arquivo .py no terminal para esta interface: ela depende do ambiente de notebook e, nos dois jogos de tabuleiro, do loop assíncrono do Colab.

In [ ]:
import random

class Tetris:
    PECAS = [ [[1,1,1,1]], [[2,2],[2,2]], [[0,3,0],[3,3,3]],
              [[0,4,4],[4,4,0]], [[5,5,0],[0,5,5]],
              [[6,0,0],[6,6,6]], [[0,0,7],[7,7,7]] ]

    def __init__(self):
        self.tabuleiro = [[0]*10 for _ in range(20)]
        self.pontos = 0
        self.linhas = 0
        self.fim = False
        self.nova_peca()

    def nova_peca(self):
        self.peca = [linha[:] for linha in random.choice(self.PECAS)]
        self.x = (10-len(self.peca[0]))//2
        self.y = 0
        if not self.cabe(self.peca, self.x, self.y):
            self.fim = True

    def cabe(self, peca, x, y):
        for dy, linha in enumerate(peca):
            for dx, valor in enumerate(linha):
                if valor and (not 0 <= x+dx < 10 or not 0 <= y+dy < 20 or self.tabuleiro[y+dy][x+dx]):
                    return False
        return True

    def mover(self, dx):
        if not self.fim and self.cabe(self.peca, self.x+dx, self.y):
            self.x += dx

    def girar(self):
        if self.fim:
            return
        girada = [list(linha) for linha in zip(*self.peca[::-1])]
        for ajuste in [0, -1, 1, -2, 2]:
            if self.cabe(girada, self.x+ajuste, self.y):
                self.peca = girada
                self.x += ajuste
                return

    def descer(self):
        if self.fim:
            return
        if self.cabe(self.peca, self.x, self.y+1):
            self.y += 1
            return
        for dy, linha in enumerate(self.peca):
            for dx, valor in enumerate(linha):
                if valor:
                    self.tabuleiro[self.y+dy][self.x+dx] = valor
        restantes = [linha for linha in self.tabuleiro if not all(linha)]
        removidas = 20-len(restantes)
        self.tabuleiro = [[0]*10 for _ in range(removidas)] + restantes
        self.linhas += removidas
        self.pontos += [0, 100, 300, 500, 800][removidas]
        self.nova_peca()

    def soltar(self):
        if self.fim:
            return
        while self.cabe(self.peca, self.x, self.y+1):
            self.y += 1
        self.descer()

if __name__ == "__main__":
    import asyncio
    import ipywidgets as w
    from IPython.display import display
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
    if "tarefa_tetris" in globals():
        tarefa_tetris.cancel()
    jogo_tetris = Tetris()
    tetris_rodando = False
    painel_tetris = w.HTML()
    cores = ["#0f172a", "#22d3ee", "#facc15", "#c084fc", "#4ade80", "#fb7185", "#60a5fa", "#fb923c"]

    def desenhar_tetris():
        j = jogo_tetris
        quadro = [linha[:] for linha in j.tabuleiro]
        if not j.fim:
            for dy, linha in enumerate(j.peca):
                for dx, valor in enumerate(linha):
                    if valor:
                        quadro[j.y+dy][j.x+dx] = valor
        blocos = "".join(f'<rect x="{x*20+1}" y="{y*20+1}" width="18" height="18" rx="2" fill="{cores[v]}"/>'
                         for y, linha in enumerate(quadro) for x, v in enumerate(linha))
        estado = "Fim de jogo" if j.fim else ("Jogando" if tetris_rodando else "Pausado")
        painel_tetris.value = f'<p><b>Tetris | Pontos: {j.pontos} | Linhas: {j.linhas}</b> | {estado}</p><svg viewBox="0 0 200 400" width="200" style="max-width:100%;background:#1e293b">{blocos}</svg>'

    def acao_tetris(acao):
        if not tetris_rodando or jogo_tetris.fim:
            return
        acao()
        desenhar_tetris()

    def pausar_tetris(_):
        global tetris_rodando
        if not jogo_tetris.fim:
            tetris_rodando = not tetris_rodando
        desenhar_tetris()

    def reiniciar_tetris(_):
        global jogo_tetris, tetris_rodando
        jogo_tetris = Tetris()
        tetris_rodando = False
        desenhar_tetris()

    controles = []
    for texto, acao in [("←", lambda: jogo_tetris.mover(-1)), ("Girar", lambda: jogo_tetris.girar()),
                         ("→", lambda: jogo_tetris.mover(1)), ("↓", lambda: jogo_tetris.descer()),
                         ("Soltar", lambda: jogo_tetris.soltar())]:
        b = w.Button(description=texto, layout=w.Layout(width="65px", height="42px"))
        b.on_click(lambda _, acao=acao: acao_tetris(acao))
        controles.append(b)
    pausa = w.Button(description="Iniciar / Pausar")
    pausa.on_click(pausar_tetris)
    reset = w.Button(description="Reiniciar")
    reset.on_click(reiniciar_tetris)

    async def atualizar_tetris():
        while True:
            await asyncio.sleep(max(0.15, 0.7-jogo_tetris.linhas*0.02))
            if tetris_rodando and not jogo_tetris.fim:
                jogo_tetris.descer()
                desenhar_tetris()

    display(w.VBox([painel_tetris, w.HBox(controles, layout=w.Layout(flex_flow="row wrap")), w.HBox([pausa, reset])]))
    desenhar_tetris()
    tarefa_tetris = asyncio.get_running_loop().create_task(atualizar_tetris())


### Desafio
Altere a pontuação de uma linha de 100 para 150 e complete uma linha.

## RPG textual
RPG interativo com botões de escolha, vida, inventário, derrota e vitória. Execute as etapas 4 e 5; encontre a chave na floresta e abra a torre. Os primeiros exemplos continuam ensinando os fundamentos. A interface abre na saída do Colab, com uma sessão conectada; estes códigos não executam dentro do site.

### Etapa 1
Uma ficha de personagem pode ser feita só com duas variáveis.

In [ ]:
nome = "Aventureiro"
vida = 10
print(nome, "tem", vida, "de vida")

### Etapa 2
A lista funciona como mochila do herói.

In [ ]:
inventario = ["mapa", "espada"]
print("Itens:", inventario)

### Etapa 3
O exemplo cria seu próprio inventário e não depende da etapa anterior. Altere escolha para "1" ou "2" para explorar os caminhos. Depois acrescente a opção "3" do desafio.

In [ ]:
inventario = ["mapa", "espada"]
escolha = "1"
if escolha == "1":
    print("Você entrou na floresta.")
elif escolha == "2":
    print("Você voltou à cidade.")
else:
    print("Escolha novamente.")
print("Itens:", inventario)

### Etapa 4
Preparação: execute esta célula no Google Colab. Se houver pedido de reinício, reinicie a sessão e execute a etapa seguinte. Esta instalação é comum aos quatro projetos.

In [ ]:
%pip install -q "ipywidgets>=8.1,<9"

### Etapa 5
RPG interativo com botões de escolha, vida, inventário, derrota e vitória. Execute as etapas 4 e 5; encontre a chave na floresta e abra a torre. Não use um arquivo .py no terminal para esta interface: ela depende do ambiente de notebook e, nos dois jogos de tabuleiro, do loop assíncrono do Colab.

In [ ]:
class Aventura:
    def __init__(self):
        self.local = "cidade"
        self.vida = 10
        self.inventario = []
        self.fim = False
        self.mensagem = "Encontre a chave na floresta e abra o portão da torre."

    def escolher(self, acao):
        if acao == "reiniciar":
            self.__init__()
            return
        if self.fim:
            return
        if self.local == "cidade":
            if acao == "floresta":
                self.local = "floresta"
                self.mensagem = "Há um baú sob uma árvore e uma trilha perigosa."
            elif acao == "descansar":
                self.vida = 10
                self.mensagem = "Você descansou. Vida recuperada!"
        elif self.local == "floresta":
            if acao == "bau":
                if "chave" not in self.inventario:
                    self.inventario.append("chave")
                    self.mensagem = "Você encontrou a chave da torre!"
                else:
                    self.mensagem = "O baú está vazio; a chave já está na mochila."
            elif acao == "trilha":
                self.vida = max(0, self.vida-3)
                self.mensagem = "Espinhos! Você perdeu 3 pontos de vida."
                if self.vida == 0:
                    self.fim = True
                    self.mensagem = "Sua aventura terminou. Clique em Reiniciar."
            elif acao == "torre":
                self.local = "torre"
                self.mensagem = "Um portão trancado protege o livro do conhecimento."
            elif acao == "cidade":
                self.local = "cidade"
                self.mensagem = "Você retornou à cidade."
        elif self.local == "torre":
            if acao == "abrir":
                if "chave" in self.inventario:
                    self.fim = True
                    self.mensagem = "Vitória! Você abriu o portão e recuperou o livro do conhecimento."
                else:
                    self.mensagem = "Você precisa encontrar a chave na floresta."
            elif acao == "floresta":
                self.local = "floresta"
                self.mensagem = "Você voltou à floresta."

    def opcoes(self):
        if self.fim:
            return []
        return {"cidade": [("Entrar na floresta", "floresta"), ("Descansar", "descansar")],
                "floresta": [("Abrir baú", "bau"), ("Explorar trilha", "trilha"), ("Ir à torre", "torre"), ("Voltar à cidade", "cidade")],
                "torre": [("Abrir portão", "abrir"), ("Voltar à floresta", "floresta")]}[self.local]

if __name__ == "__main__":
    import html
    import ipywidgets as w
    from IPython.display import display
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
    aventura = Aventura()
    painel_rpg = w.HTML()
    escolhas_rpg = w.VBox()

    def jogar_rpg(acao):
        aventura.escolher(acao)
        desenhar_rpg()

    def desenhar_rpg():
        a = aventura
        painel_rpg.value = f'<h3>RPG — O livro do conhecimento</h3><p><b>Local:</b> {a.local} | <b>Vida:</b> {a.vida}/10</p><p><b>Mochila:</b> {html.escape(", ".join(a.inventario) or "vazia")}</p><p>{html.escape(a.mensagem)}</p>'
        botoes = []
        for texto, acao in a.opcoes() + [("Reiniciar", "reiniciar")]:
            b = w.Button(description=texto, layout=w.Layout(width="220px", height="40px"))
            b.on_click(lambda _, acao=acao: jogar_rpg(acao))
            botoes.append(b)
        escolhas_rpg.children = tuple(botoes)

    display(w.VBox([painel_rpg, escolhas_rpg]))
    desenhar_rpg()


### Desafio
Acrescente um item ao baú e faça esse item aparecer na mochila.

## Bot clássico ELIZA
ELIZA didática com conversa interativa, padrões de frases e reflexão de pronomes. Execute as etapas 4 e 5, digite uma mensagem e clique em Enviar. Os primeiros exemplos continuam ensinando os fundamentos. A interface abre na saída do Colab, com uma sessão conectada; estes códigos não executam dentro do site.

### Etapa 1
lower converte para minúsculas; split separa palavras por espaços. Procurar "oi" na lista evita confundir a saudação com parte da palavra "coisa". Neste primeiro exemplo, use palavras sem pontuação.

In [ ]:
frase = "Oi professor"
palavras = frase.lower().split()
if "oi" in palavras:
    print("Olá! Vamos estudar Python?")

### Etapa 2
re é um módulo da biblioteca padrão: não precisa de pip. Ele separa palavras mesmo com pontuação. A função responde a "Oi!", mas não confunde "coisa" com "oi".

In [ ]:
import re

def responder(frase):
    palavras = re.findall(r"\w+", frase.lower())
    if "lista" in palavras:
        return "Use [ ] para criar uma lista."
    if "oi" in palavras:
        return "Olá, estudante!"
    return "Me conte mais."

print(responder("como criar lista?"))
print(responder("Oi!"))
print(responder("coisa"))

### Etapa 3
Cada pergunta passa pela função. As regras reconhecem palavras completas, ignorando maiúsculas e pontuação. Este é um bot didático inspirado em regras clássicas, não a implementação original de ELIZA.

In [ ]:
import re

def bot(frase):
    palavras = re.findall(r"\w+", frase.lower())
    if "erro" in palavras:
        return "Leia a última linha do erro."
    if "python" in palavras:
        return "Python é uma linguagem de programação."
    return "Quer estudar listas ou jogos?"

perguntas = ["O que é Python?", "Deu erro"]
for pergunta in perguntas:
    print("Você:", pergunta)
    print("Bot:", bot(pergunta))

### Etapa 4
Preparação: execute esta célula no Google Colab. Se houver pedido de reinício, reinicie a sessão e execute a etapa seguinte. Esta instalação é comum aos quatro projetos.

In [ ]:
%pip install -q "ipywidgets>=8.1,<9"

### Etapa 5
ELIZA didática com conversa interativa, padrões de frases e reflexão de pronomes. Execute as etapas 4 e 5, digite uma mensagem e clique em Enviar. Não use um arquivo .py no terminal para esta interface: ela depende do ambiente de notebook e, nos dois jogos de tabuleiro, do loop assíncrono do Colab.

In [ ]:
import re
import unicodedata

def normalizar(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto.lower()) if not unicodedata.combining(c))

def refletir(texto):
    trocas = {"eu": "você", "meu": "seu", "minha": "sua", "meus": "seus", "minhas": "suas",
              "sou": "é", "estou": "está", "tenho": "tem", "voce": "eu", "seu": "meu", "sua": "minha"}
    return re.sub(r"\b\w+\b", lambda m: trocas.get(normalizar(m.group()), m.group()), texto)

class Eliza:
    def __init__(self):
        self.encerrada = False
        self.turno = 0

    def responder(self, texto):
        if self.encerrada:
            return "Conversa encerrada. Clique em Nova conversa."
        texto = texto.strip()
        if not texto:
            return "Escreva uma frase para começarmos."
        n = normalizar(texto)
        if n in ["sair", "tchau", "adeus"]:
            self.encerrada = True
            return "Até mais! Obrigada pela conversa."
        if re.search(r"\b(oi|ola)\b", n):
            return "Olá! Sou uma ELIZA didática, um programa de regras. Sobre o que quer conversar?"
        for padrao, resposta in [
            (r"\beu preciso (.+)", "Por que você precisa {trecho}?"),
            (r"\beu sinto (.+)", "O que faz você sentir {trecho}?"),
            (r"\beu estou (.+)", "Há quanto tempo você está {trecho}?"),
            (r"\beu sou (.+)", "O que significa para você ser {trecho}?"),
            (r"\bporque (.+)", "Essa é a única razão: {trecho}?"),
        ]:
            achado = re.search(padrao, n)
            if achado:
                # A normalização mantém os índices das letras portuguesas usuais.
                trecho = texto[achado.start(1):achado.end(1)].rstrip(".!?")
                trecho = refletir(trecho)
                if "precisa {trecho}" in resposta and not re.match(r"^(de|do|da|dos|das)\b", normalizar(trecho)):
                    trecho = "de " + trecho
                return resposta.format(trecho=trecho)
        if re.search(r"\b(mae|pai|familia|irmao|irma)\b", n):
            return "Pode me contar mais sobre sua família?"
        if re.search(r"\b(python|programacao|codigo)\b", n):
            return "Que parte da programação você está aprendendo?"
        respostas = ["Pode dar um exemplo?", "O que você pensa sobre isso?", "Conte um pouco mais."]
        resposta = respostas[self.turno % len(respostas)]
        self.turno += 1
        return resposta

if __name__ == "__main__":
    import html
    import ipywidgets as w
    from IPython.display import display
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
    eliza = Eliza()
    historico_eliza = []
    conversa_eliza = w.HTML()
    entrada_eliza = w.Textarea(description="Você:", placeholder="Digite sua mensagem…", layout=w.Layout(width="95%"))
    enviar_eliza = w.Button(description="Enviar")
    nova_eliza = w.Button(description="Nova conversa")

    def atualizar_eliza():
        conversa_eliza.value = '<h3>ELIZA didática</h3><p>Bot por regras inspirado em ELIZA. Não é uma LLM nem um serviço de aconselhamento.</p><div style="max-height:300px;overflow:auto">' + "".join(
            f'<p><b>{autor}:</b> {html.escape(texto)}</p>' for autor, texto in historico_eliza[-60:]) + '</div>'

    def enviar_mensagem(_):
        texto = entrada_eliza.value.strip()
        if not texto:
            return
        historico_eliza.append(("Você", texto))
        historico_eliza.append(("ELIZA", eliza.responder(texto)))
        entrada_eliza.value = ""
        atualizar_eliza()

    def nova_conversa(_):
        global eliza
        eliza = Eliza()
        historico_eliza.clear()
        historico_eliza.append(("ELIZA", "Olá! Digite uma frase. Para encerrar, escreva sair."))
        entrada_eliza.value = ""
        atualizar_eliza()

    enviar_eliza.on_click(enviar_mensagem)
    nova_eliza.on_click(nova_conversa)
    display(w.VBox([conversa_eliza, entrada_eliza, w.HBox([enviar_eliza, nova_eliza])]))
    nova_conversa(None)


### Desafio
Acrescente uma regra para escola e teste com duas frases.

## Minha primeira LLM no Colab
Esta etapa exige internet e uma sessão do Google Colab. O site continua sendo um tutor por regras: a LLM será executada somente no Colab. Começaremos carregando um modelo pequeno e limitado, sem treinar um modelo do zero. O SmolLM2-135M foi escolhido pelo tamanho; trabalha principalmente em inglês e pode responder mal em português. Cada etapa após a instalação contém seus próprios imports e carrega o modelo, sem depender de variáveis de outra célula.

### Etapa 1
Execute esta célula de instalação no Google Colab antes das etapas 2 ou 3. %pip instala no ambiente do notebook; não funciona em arquivo .py. Se o Colab pedir, reinicie a sessão e prossiga. Fixamos Transformers 4.56.2 para usar a mesma API da aula.

In [ ]:
%pip install -q "transformers==4.56.2" torch

### Etapa 2
Após a instalação da etapa 1, esta célula importa a biblioteca e baixa o modelo. Funciona na CPU, sem exigir GPU. Saída esperada: Modelo carregado na CPU. O download inicial precisa de internet.

In [ ]:
from transformers import pipeline

gerador = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-135M-Instruct",
    device=-1,
)
print("Modelo carregado na CPU.")

### Etapa 3
Após instalar na etapa 1, cole este exemplo inteiro em uma célula. Ele carrega o modelo, envia a pergunta e mostra a resposta do assistente. A pergunta em inglês combina com o modelo; depois teste português e compare. Não existe uma resposta exata garantida.

In [ ]:
from transformers import pipeline

gerador = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-135M-Instruct",
    device=-1,
)

mensagens = [
    {"role": "user", "content": "Explain a Python list in two short sentences."}
]
saida = gerador(
    mensagens,
    max_new_tokens=80,
    do_sample=False,
    pad_token_id=gerador.tokenizer.eos_token_id,
)
resposta = saida[0]["generated_text"][-1]["content"]
print(resposta)

### Desafio
No Colab, modifique a pergunta e compare a resposta da LLM com uma resposta do seu bot de regras. Registre três diferenças.